In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
ROOT = Path.cwd() if (Path.cwd() / "PLAN.md").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.ingest import FILES, missing_files
missing = missing_files()
if missing:
    raise FileNotFoundError("Place these files in data/raw/: " + ", ".join(missing))
tables = {name: pd.read_csv(ROOT / "data/raw" / filename, low_memory=False) for name, filename in FILES.items()}


In [ ]:
# Every table: rows, inferred types, nulls, duplicate rows, top values, and numeric ranges.
profile = {}
for name, df in tables.items():
    profile[name] = {
        "rows": len(df),
        "types": {column: str(dtype) for column, dtype in df.dtypes.items()},
        "nulls": df.isna().sum().astype(int).to_dict(),
        "duplicate_rows": int(df.duplicated().sum()),
        "top_values": {column: df[column].value_counts(dropna=False).head(5).astype(int).to_dict()
                       for column in df.select_dtypes(include="object").columns},
        "numeric_distributions": {column: {str(q): float(df[column].quantile(q)) for q in (0, .25, .5, .75, 1)}
                                  for column in df.select_dtypes(include="number").columns if df[column].notna().any()},
        "date_ranges": {column: [str(pd.to_datetime(df[column], errors="coerce").min()), str(pd.to_datetime(df[column], errors="coerce").max())]
                        for column in df.columns if "date" in column or "timestamp" in column},
    }
    print(name, "rows:", len(df), "duplicate rows:", profile[name]["duplicate_rows"])
    display(pd.DataFrame({"type": df.dtypes.astype(str), "nulls": df.isna().sum()}))


In [ ]:
# Date ranges, sparse months, and documented Olist quirks.
orders = tables["orders"]
purchase = pd.to_datetime(orders.order_purchase_timestamp, errors="coerce")
monthly = purchase.dt.to_period("M").value_counts().sort_index()
profile["orders"]["purchase_range"] = [str(purchase.min()), str(purchase.max())]
profile["orders"]["monthly_counts"] = {str(k): int(v) for k, v in monthly.items()}
reliable = monthly[monthly >= 500]
profile["analysis_window_provisional"] = [str(reliable.index.min()), str(reliable.index.max())]
items = tables["order_items"]
payments = tables["order_payments"]
reviews = tables["order_reviews"]
products = tables["products"]
translation = tables["category_translation"]
profile["quirks"] = {
    "orders_without_items": int((~orders.order_id.isin(items.order_id)).sum()),
    "orders_without_payments": int((~orders.order_id.isin(payments.order_id)).sum()),
    "duplicate_review_ids": int(reviews.review_id.duplicated().sum()),
    "orders_with_multiple_reviews": int((reviews.groupby("order_id").size() > 1).sum()),
    "duplicate_zip_prefixes": int(tables["geolocation"].geolocation_zip_code_prefix.duplicated().sum()),
    "untranslated_categories": int((~products.product_category_name.dropna().isin(translation.product_category_name)).sum()),
    "delivered_missing_delivery_date": int(((orders.order_status == "delivered") & orders.order_delivered_customer_date.isna()).sum()),
}
print(monthly.to_string())
print(profile["quirks"])
print("Provisional analysis window:", profile["analysis_window_provisional"])


In [ ]:
# Save an auditable local profile; transfer the observed values to docs/data_dictionary.md.
out = ROOT / "data/profile.json"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(profile, indent=2, default=str, ensure_ascii=False))
print(out)
